## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## This Notebook:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

In [1]:
# Run this line if not already Installed:
#!pip install langchain-huggingface sentence-transformers
#!pip install nbformat

In [2]:
# Imports
import os
import glob
import tiktoken
from dotenv import load_dotenv
from huggingface_hub import login, HUGGINGFACE_CO_URL_HOME
from langchain_classic import text_splitter
from openai import OpenAI
import numpy as np

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

## Connecting to OpenAI API and HuggingFace:

In [3]:
MODEL = 'gpt-4.1-nano'
db_name = 'vector_db'

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')

openai = OpenAI()
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Part A: Dividing Documents into Chunks

In [4]:
# Counting Characters in all Documents:

knowledge_base_path = 'knowledge-base/**/*.md'
files = glob.glob(knowledge_base_path, recursive= True)
print(f'Found {len(files)} files in Knowledge Base.')

entire_knowledge_base = ''

for file_path in files:
    with open(file_path, 'r', encoding= 'utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += '\n\n'

print(f'Total Characters in Knowledge Base: {len(entire_knowledge_base)}')

Found 76 files in Knowledge Base.
Total Characters in Knowledge Base: 304434


In [5]:
# Counting Number of Tokens in All Documents:

encoding = tiktoken.encoding_for_model(model_name= MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f'Total tokens for {MODEL}: {token_count}')

Total tokens for gpt-4.1-nano: 63555


### Loading in Everything in the Knowledge Base Using Langchain's Loaders:

In [6]:
folders = glob.glob('knowledge-base/*')
documents = []
# print(folders)

for folder in folders:
    # Setting up Document-Type as Folder Name:
    doc_type = os.path.basename(folder)

    # Directory Loader Instance:
    loader = DirectoryLoader(folder, glob= '**/*.md', loader_cls= TextLoader, loader_kwargs= {'encoding': 'utf-8'})

    # Loading Data:
    folder_docs = loader.load()

    # Adding Everything to One List:
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents.')

Loaded 76 documents.


In [7]:
documents[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a

### Dividing Documents into Chunks:

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap= 200)
chunks = text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks.')

Divided into 413 chunks.


In [9]:
chunks[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

In [10]:
chunks[1]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content='However, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active contracts spanning all eight product lines. The company maintains its San Francisco headquarters along with small satellite offices in key markets including New York, Austin, Chicago, and Denver.\n\nSince the restructuring, Insurellm has continued to innovate, expanding its product suite to eight comprehensive platforms. The company added Lifellm (life insurance), Healthllm (health insurance), Bizllm (commercial insurance), and Claimllm (claims processing) to serve the full spectrum of insurance needs. This strategic expansion has been

## PART B: Making Vectors and Storing them in Chroma

### Picking an Embedding Model:

In [11]:
embeddings = HuggingFaceEmbeddings(model_name= 'all-MiniLM-L6-v2')
#embeddings = HuggingFaceEmbeddings(model_name= 'text-embedding-3-Small')
#embeddings = HuggingFaceEmbeddings(model_name= 'text-embedding-3-Large')

# Wiping database if already exists:
if os.path.exists(db_name):
    Chroma(persist_directory= db_name, embedding_function= embeddings).delete_collection()

# Creating Vector Database using Chroma:
vectorstore = Chroma.from_documents(documents= chunks,
                                    embedding= embeddings,
                                    persist_directory= db_name)

print(f'VectorStore Created with {vectorstore._collection.count()} documents.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStore Created with 413 documents.


In [12]:
# Investigating Vectors:

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit= 1, include= ['embeddings'])['embeddings'][0]
dimensions = len(sample_embedding)
print(f'There are {count} Vectors with {dimensions} dimensions.')

There are 413 Vectors with 384 dimensions.


In [13]:
sample_embedding

array([-1.97564941e-02, -2.53404714e-02, -5.02225272e-02,  4.75930162e-02,
       -5.27324341e-03, -8.04971438e-03,  1.04124144e-01,  3.71347591e-02,
       -2.81660259e-02, -3.68553028e-02,  5.98170310e-02,  4.09410000e-02,
        9.58695039e-02, -3.57764512e-02, -1.49062229e-02, -5.74225141e-03,
       -2.41792221e-02, -4.07417901e-02,  3.82803418e-02, -4.92585357e-03,
       -4.23193164e-02, -3.38826329e-03, -2.81992070e-02, -1.62199978e-02,
       -1.09575214e-02, -1.64638646e-02,  5.98575026e-02,  6.27911165e-02,
       -3.15549113e-02, -5.67085780e-02,  9.01479460e-03,  3.04025109e-03,
        6.17995160e-03,  3.58186737e-02,  3.75843560e-03,  2.58685462e-02,
       -5.76471053e-02, -7.67623410e-02, -9.59911272e-02, -1.94395203e-02,
        3.65776569e-02, -2.07921062e-02, -5.12749888e-02, -1.58227421e-02,
       -7.07919970e-02,  2.56839283e-02,  2.65006255e-02,  6.56482652e-02,
       -5.28969765e-02,  1.45103037e-01, -1.00828698e-02,  7.10691959e-02,
        9.63132270e-03, -

## PART C: Visualizing Vecotrs:

In [14]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [15]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components= 2, random_state= 42)
reduced_vectors = tsne.fit_transform(vectors)

### 2D-Scatter Plot:

In [16]:
fig = go.Figure(data=[go.Scatter(
    x= reduced_vectors[:, 0],
    y= reduced_vectors[:, 1],
    mode= 'markers',
    marker= dict(size=5, color=colors, opacity=0.8),
    text= [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo= 'text'
)])

fig.update_layout(title= '2D Chroma Vector Store Visualization',
    scene= dict(xaxis_title='x',yaxis_title='y'),
    width= 800,
    height= 600,
    margin= dict(r=20, b=10, l=10, t=40)
)

fig.show()

### 3D-Scatter Plot:

In [17]:
tsne = TSNE(n_components= 3, random_state= 42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x= reduced_vectors[:, 0],
    y= reduced_vectors[:, 1],
    z= reduced_vectors[:, 2],
    mode= 'markers',
    marker= dict(size=5, color=colors, opacity=0.8),
    text= [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo= 'text'
)])

fig.update_layout(
    title= '3D Chroma Vector Store Visualization',
    scene= dict(xaxis_title= 'x', yaxis_title= 'y', zaxis_title= 'z'),
    width= 900,
    height= 700,
    margin= dict(r= 10, b= 10, l= 10, t= 40)
)

fig.show()